In [1]:
from langgraph.types import Interrupt, Command
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import HumanMessage, AIMessage
from operator import add
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
class AgentState(TypedDict):
    action:str
    result:str
    approval:bool


In [3]:
def start(state:AgentState)->AgentState:
    return {
        "action":"Delete Database",
        "approval":False,
        "result":""
    }

In [4]:
def human_approval(state:AgentState)->AgentState:
    print("\n=== WORKFLOW PAUSED ===")
    print(f"Action requested: {state['action']}")
    print("Waiting for user approval...")
    
    answere = Interrupt(value="Are you sure you want to delete the database?")
    
    if answere:
        print("User approved. Proceeding...")
    else:
        print("User declined. Cancelling...")
    
    return {"approval": answere}

In [5]:
def execute(state:AgentState)->AgentState:
    
    ans=state['approval']
    
    if ans==True:

        return {
            "result":
            "Database deleted"
        }


    return {

        "result":
        "Action cancelled"

    }

In [6]:
graph=StateGraph(AgentState)

In [7]:
graph.add_node("start",start)
graph.add_node("human_approval",human_approval)
graph.add_node("execute",execute)

graph.add_edge(START,"start")
graph.add_edge("start","human_approval")
graph.add_edge("human_approval","execute")
graph.add_edge("execute",END)

In [8]:
memory=InMemorySaver()

In [9]:
app=graph.compile(checkpointer=memory)

In [10]:
config = {

"configurable":{

"thread_id":"user1"

}

}


In [13]:
try:
    result = app.invoke(
    {
        "action":"",
        "approval":False,
        "result":""
    },
    config
    )
    print(result)
except Exception as e:
    print("Graph interrupted. Resume with approval value:", e)


=== WORKFLOW PAUSED ===
Action requested: Delete Database
Waiting for user approval...
User approved. Proceeding...
{'action': 'Delete Database', 'result': 'Action cancelled', 'approval': Interrupt(value='Are you sure you want to delete the database?', id='placeholder-id')}


In [14]:
# === Resume the workflow with user decision ===
# Change to Command(resume=False) to cancel
result = app.invoke(
    Command(resume=True),
    config
)
print(result)

{'action': 'Delete Database', 'result': 'Action cancelled', 'approval': Interrupt(value='Are you sure you want to delete the database?', id='placeholder-id')}


In [15]:
# Check the final state
state = app.get_state(config)
print(state.values)

{'action': 'Delete Database', 'result': 'Action cancelled', 'approval': Interrupt(value='Are you sure you want to delete the database?', id='placeholder-id')}
